In [ ]:
import requests, re, random
from config import HF_API_KEY # Imports the Hugging Face API key from a config file

# MODEL variable defines the specific sentence transformer model to be used.
# The commented-out line shows an alternative, more powerful model if required.
MODEL="sentence-transformers/all-MiniLM-L6-v2"

# API endpoint URL for the Hugging Face Inference API.
API=f"https://router.huggingface.co/hf-inference/models/{MODEL}"

# Authorization header including the API key for secure requests.
HEAD={"Authorization":f"Bearer {HF_API_KEY}"}

# Threshold for determining if two sentences are duplicates (similarity score).
TH=0.72

# A list of example sentence pairs for demonstration purposes.
DEMOS=[("how to delete my account","how do i remove my account"),
       ("start the game","begin the game"),
       ("nearest hospital to me","closest clinic near me"),
       ("mobile games are getting bigger in size","game size on phones is increasing"),
       ("is it going to rain today","today is rainy"),
       ("reset my password","change my password")]

# Lambda function to tokenize a string by splitting and joining with ' | '.
TOK=lambda s:" | ".join(s.split())

# Lambda function to create a progress bar string based on a similarity score.
bar=lambda s:"█"*int(s*10)+"░"*(10-int(s*10))

# Lambda function to clean a string: converts to lowercase, removes non-alphanumeric characters,
# and filters out empty words.
clean=lambda t:[w for w in (re.sub(r"[^a-z0-9']+","",x.lower()) for x in t.split()) if w]

# Lambda function to extract all numbers (integers or floats) from a string.
nums=lambda t:set(re.findall(r"\d+(?:\.\d+)?", t))

# Lambda function to check if any word from a given array is present in a cleaned string.
has_any=lambda t,arr:any(a in set(clean(t)) for a in arr)

# Function to perform a similarity query using the Hugging Face API.
def hf(q1,q2):
    # Sends a POST request to the HF API with the two sentences.
    r=requests.post(API,headers=HEAD,json={"inputs":{"source_sentence":q1,"sentences":[q2]}},timeout=30)
    # Raises an error if the API request was not successful.
    if not r.ok: raise RuntimeError(r.text)
    # Parses the JSON response.
    data=r.json()
    # Raises an error if the response contains an error message.
    if isinstance(data,dict): raise RuntimeError(data.get("error",str(data)))
    # Returns the similarity score as a float.
    return float(data[0])

# Function to calculate a 'smart score' by adjusting the base similarity score
# with boosts and penalties based on various linguistic features.
def smart_score(base,q1,q2,strong):
    # Extracts words of length 4 or more from cleaned sentences for Jaccard index calculation.
    w1={w for w in clean(q1) if len(w)>=4}; w2={w for w in clean(q2) if len(w)>=4}
    # Calculates the Jaccard similarity index for the 'strong' words.
    jac=len(w1&w2)/max(1,len(w1|w2))
    # Calculates a boost based on the number of strong shared words and Jaccard index.
    boost=(0.04 if len(strong)>=2 else 0)+(0.03 if jac>=0.20 else 0)+(0.05 if jac>=0.35 else 0)
    # Lists of negative words and opposing word pairs for penalty calculations.
    negA=["not","no","never","without","can't","cant","cannot","don't","dont","won't","wont","n't"]
    oppA=[("increase","decrease"),("bigger","smaller"),("more","less"),("add","remove"),("open","close"),("enable","disable")]
    # Penalizes if numbers are present and differ between sentences.
    num_pen=0.10 if (nums(q1) and nums(q2) and nums(q1)!=nums(q2)) else 0
    # Penalizes if one sentence has a negative word and the other doesn't.
    neg_pen=0.12 if has_any(q1,negA)!=has_any(q2,negA) else 0
    # Penalizes if opposing words are found across the two sentences.
    opp_pen=0.12 if any((has_any(q1,[a]) and has_any(q2,[b])) or (has_any(q1,[b]) and has_any(q2,[a])) for a,b in oppA) else 0
    # Returns the final adjusted score, clamped between 0.0 and 1.0.
    return max(0.0, min(1.0, base+boost-num_pen-neg_pen-opp_pen))

# Function to categorize the similarity score into 'DUPLICATE', 'CLOSE MATCH', or 'DIFFERENT'.
def label(s): return "✅ DUPLICATE" if s>=TH else ("🤔 CLOSE MATCH" if s>=TH-0.05 else "❌ DIFFERENT")

# Function to display the similarity result and the rule.
def show_result(s):
    print(f"\n🎯 Result of Similarity: {round(s*100,1)}% [{bar(s)}]  →  {label(s)}")
    print(f"Rule: score ≥ {TH} means DUPLICATE")

# Function to show the flow of how similarity is determined, including word breakdown.
def show_flow(q1,q2):
    # Cleans and combines words from both questions.
    a,b=clean(q1),clean(q2); raw=set(a+b)
    # Extracts 'strong' words (length >= 4) from each question.
    w1={w for w in a if len(w)>=4}; w2={w for w in b if len(w)>=4}
    # Finds shared strong words.
    shared=sorted(w1&w2)
    # Finds helper words (length 2 or 3).
    helpers=sorted({w for w in raw if 2<=len(w)<=3})
    # Defines common connective words.
    conn={"a","an","the","to","of","in","on","is","am","are","do","did","does","my","me","it"}
    # Finds least significant words (length <= 2 or common connectives).
    least=sorted({w for w in raw if len(w)<=2 or w in conn})
    print("\n🔁 FLOW (sentence → strongest/helper/least → similarity %)")
    print("\n1) Input sentences"); print(f"   Q1: {q1}\n   Q2: {q2}")
    print("\n2) Split into words/tokens (same as you typed)")
    print("   Q1 →",TOK(q1)); print("   Q2 →",TOK(q2))
    print("\n3) Pick the “meaning-carrying” parts (from YOUR sentences)")
    print("   Strongest:",", ".join(shared) if shared else "No obvious shared/synonym matches")
    print("   Helper:",", ".join(helpers) if helpers else "None")
    print("   Least:",", ".join(least) if least else "None")
    print("\n4) Why similarity is high/low for THIS pair")
    # Explains if direct matches were found or if the model relied on overall meaning.
    print("   - Direct matches:",", ".join(shared)) if shared else print("   - The model used overall meaning patterns, not exact word matches.")

# Main function to run the similarity comparison for a pair of questions.
def run(q1,q2,title):
    print(f"\n--- {title} ---")
    # Gets the base similarity score from the Hugging Face API.
    base=hf(q1,q2)
    # Finds strong words shared between the cleaned questions.
    strong=sorted({w for w in clean(q1) if len(w)>=4} & {w for w in clean(q2) if len(w)>=4})
    # Calculates the smart score.
    s=smart_score(base,q1,q2,strong)
    # Displays the result and the flow.
    show_result(s); show_flow(q1,q2)

# Entry point of the script, handling user interaction.
def main():
    print("Type Question 1 → Question 2. Then you’ll see 2 RANDOM demo pairs.")
    print("Type 'exit' anytime to quit.\n")
    while True:
        q1=input("Question 1: ").strip()
        if q1.lower()=="exit": break # Exits if user types 'exit'.
        q2=input("Question 2: ").strip()
        if q2.lower()=="exit": break # Exits if user types 'exit'.
        if not q1 or not q2: continue # Continues to next iteration if inputs are empty.
        try:
            run(q1,q2,"YOUR QUESTIONS") # Runs the similarity check for user's questions.
            # Runs similarity checks for two random demo pairs.
            for i,(d1,d2) in enumerate(random.sample(DEMOS,2),1): run(d1,d2,f"RANDOM DEMO {i}")
            print("\n(Next round → Question 1 or 'exit')\n")
        except Exception as e:
            print("\n⚠️ Oops!",e,"\n") # Prints an error message if an exception occurs.

# Ensures main() is called only when the script is executed directly.
if __name__=="__main__": main()